In [5]:
import pandas as pd

def debug_prompt_text(input_file):
    # 데이터 로드
    df = pd.read_csv(input_file)
    # 중복 제거 (실제 파이프라인과 동일하게)
    unique_papers = df.drop_duplicates(subset=['논문ID']).head(3)
    
    print("="*30)
    print("🚀 [DEBUG] API 전송 예정 텍스트 샘플")
    print("="*30)
    
    for i, (_, row) in enumerate(unique_papers.iterrows()):
        # 실제 classify_batch 함수에서 만드는 것과 동일한 로직
        items_text = f"- 제목: {row['제목']} / 키워드: {row['키워드']}\n"
        
        print(f"[{i+1}번 논문 전송 텍스트]:")
        print(items_text)
        print("-" * 30)

# 실행 (상세데이터 파일명을 넣으세요)
debug_prompt_text('법학_AI_논문_상세정보_리스트.csv')

🚀 [DEBUG] API 전송 예정 텍스트 샘플
[1번 논문 전송 텍스트]:
- 제목: 인공지능(AI) 활용 저작물의 편집저작물성과 구성요소의 선택·배열 / 키워드: 편집, Edit, 저작권 등록, Artificial Intelligent, Copyright Registration, 자연인 작성, Originality, 구성요소의 선택·배열, 편집물, 인공지능, 편집저작물, Compilation, 창작성, Compilation Work, Human Authorship

------------------------------
[2번 논문 전송 텍스트]:
- 제목: 국내 법제상 로봇에 대한 규제 연구 / 키워드: 자율주행로봇, Industrial Robot, 산업용 로봇, Robot, Autonomous Driving Robot, 규제, 웨어러블 로봇, 의료용 착용형 로봇, Medical Wearable Robot, Wearable Robot, 로봇, Regulation

------------------------------
[3번 논문 전송 텍스트]:
- 제목: 생성형 AI와 편향성 / 키워드: 대표성 없는 데이터, 투명성, 차별, halucinations, unrepresentative data, 생성형 인공지능, transparency, bias, 환각, accountability, inclusivity., fairness, 설명가능성, 포용성, Generative Artificial Intelligence, 공정성, 편향성

------------------------------


### LLM으로 분류

In [ ]:
import pandas as pd
import google.generativeai as genai
from pydantic import BaseModel
from typing import List
from tqdm import tqdm
from enum import Enum
import time
import os
import json
from dotenv import load_dotenv
from typing import List, Literal

import pandas as pd
from google import genai  # ⭐ 변경!
from google.genai import types  # ⭐ 추가
from pydantic import BaseModel
from typing import List, Literal
from tqdm import tqdm
import time
import os
import json
from dotenv import load_dotenv

# 0. 설정 로드
load_dotenv()
API_KEY = os.getenv("GOOGLE_API_KEY")

print(f"현재 작업 경로: {os.getcwd()}")
print(f"✅ API 키 로드: {API_KEY[:4]}..." if API_KEY else "❌ API 키 없음")

# 1. Pydantic 모델 (동일)
class ClassificationItem(BaseModel):
    source_id: str
    category: Literal["형사법", "공법", "민사법", "기타"]

class BatchClassificationResponse(BaseModel):
    results: List[ClassificationItem]

# 2. 새 SDK로 API 설정
client = genai.Client(api_key=API_KEY)

# API 연결 테스트
try:
    print("🔌 API 연결 테스트 중...")
    test_response = client.models.generate_content(
        model='gemini-2.5-flash',  # 모델명 확인
        contents='테스트'
    )
    print(f"✅ API 연결 성공! 응답: {test_response.text[:30]}")
except Exception as e:
    print(f"❌ API 연결 실패: {e}")
    exit(1)

def classify_batch(batch_df):
    """20개의 논문을 하나의 프롬프트로 묶어 분류 요청"""
    items_text = ""
    for _, row in batch_df.iterrows():
        raw_keywords = str(row['키워드'])
        clean_keywords = raw_keywords.replace(']]>', '').strip()
        items_text += f"[ID: {row['source_id']}] 제목: {row['제목']} / 키워드: {clean_keywords}\n"
    
    print(f"📏 프롬프트 길이: {len(items_text)} 글자")
    
    prompt = f"""
    <역할>
    30년 이상의 경력을 지닌 대한민국의 법학가로서, 너는 법률 논문을 분류하는 작업을 진행해야 해.
    주어진 논문들의 제목과 키워드를 읽고 각 논문을 <분류 기준>을 따라 4개 카테고리 중 하나로 분류해줘.
    </역할>

    <분류 기준>
    1. 형사법: 형벌에 관한 사항을 규율하는 법 체계

    2. 공법: 국가, 지방자치단체 등 공적 기관 상호 간, 또는 국가·지방자치단체와 개인(사인) 사이의 관계를 규율하는 법 체계 중 형사법이 아닌 것 

    3. 민사법: 개인(사인) 간의 재산적·신분적 생활 관계를 규율하는 법 체계

    4. 기타: 위 세가지 분류 중 어느 하나에 명확하게 속하지 않거나 동시에 여러 분류에 속하는 법 체계
    </분류 기준>

    [대상 리스트]
    {items_text}
    
    각 논문에 대해 source_id와 category를 JSON 형식으로 반환해.
    """

    try:
        print(f"🔄 API 호출 시작... (배치 크기: {len(batch_df)})")
        start_time = time.time()
        
        response = client.models.generate_content(
            model='gemini-2.5-flash',
            contents=prompt,
            config=types.GenerateContentConfig(
                response_mime_type="application/json",
                response_schema=BatchClassificationResponse,
                temperature=0.0
            )
        )
        
        elapsed = time.time() - start_time
        print(f"✅ API 응답 완료 ({elapsed:.2f}초)")
        
        return response.text
        
    except Exception as e:
        print(f"❌ API 에러: {type(e).__name__} - {str(e)}")
        time.sleep(10)
        return None


def run_classification_pipeline(input_file, output_file):
    print(f"📋 {input_file} 로드 중...")
    df = pd.read_csv(input_file)
    
    if '논문ID' in df.columns:
        df = df.rename(columns={'논문ID': 'source_id'})
    
    unique_papers = df.drop_duplicates(subset=['source_id']).copy()

    print(f"✅ 총 {len(unique_papers)}건의 논문을 분류합니다.")
    print(f"📦 배치 크기: 20개씩, 총 {(len(unique_papers) + 19) // 20}개 배치")
    # 메인 실행 전에 추가
    print("🧪 첫 배치 테스트...")
    test_batch = unique_papers.head(5)  # 5개만 테스트
    test_result = classify_batch(test_batch)
    print(f"테스트 결과: {test_result[:200] if test_result else 'None'}")

    batch_size = 20
    success_count = 0
    fail_count = 0
    
    for i in tqdm(range(0, len(unique_papers), batch_size), desc="LLM 분류 진행 중"):
        batch_df = unique_papers.iloc[i : i + batch_size]
        print(f"\n--- 배치 {i//batch_size + 1} 시작 (논문 {i+1}~{min(i+batch_size, len(unique_papers))}) ---")
        
        json_response = classify_batch(batch_df)
        
        if json_response:
            try:
                parsed_data = json.loads(json_response)
                batch_results = pd.DataFrame(parsed_data['results'])
                
                batch_merged = batch_results.merge(
                    batch_df[['source_id', '제목', '발행연도', '키워드']],
                    on='source_id'
                )
                
                header = not os.path.exists(output_file)
                batch_merged.to_csv(output_file, index=False, mode='a', 
                                   header=header, encoding='utf-8-sig')
                
                success_count += len(batch_df)
                print(f"✅ 저장 완료 ({len(batch_df)}건)")
                
            except Exception as e:
                fail_count += len(batch_df)
                print(f"❌ 파싱 에러: {e}")
        else:
            fail_count += len(batch_df)
            print(f"⚠️ 이 배치는 건너뜁니다.")
        
        time.sleep(1)
    
    print(f"\n{'='*60}")
    print(f"🎯 처리 완료: 성공 {success_count}건 / 실패 {fail_count}건")
    print(f"{'='*60}")
    
    if os.path.exists(output_file):
        final_df = pd.read_csv(output_file)
        print(f"\n🏁 최종 결과 저장: {output_file}")
        print(final_df['category'].value_counts())

if __name__ == "__main__":

    run_classification_pipeline(
        input_file='법학_AI_논문_상세정보_리스트.csv', 
        output_file='AI_논문_분야별_분류_최종(예시x).csv'
    )

C:\Users\silve\AppData\Local\Temp\ipykernel_44404\3406191298.py:2: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


현재 작업 경로: c:\Users\silve\Documents\GitHub\kci_network
✅ API 키 로드: AIza...
🔌 API 연결 테스트 중...
✅ API 연결 성공! 응답: 네, 안녕하세요! 잘 작동하고 있습니다. 무엇을 도와드
📋 법학_AI_논문_상세정보_리스트.csv 로드 중...
✅ 총 1903건의 논문을 분류합니다.
📦 배치 크기: 20개씩, 총 96개 배치
🧪 첫 배치 테스트...
📏 프롬프트 길이: 1255 글자
🔄 API 호출 시작... (배치 크기: 5)
✅ API 응답 완료 (6.53초)
테스트 결과: {"results": [{"source_id": "ART003212210", "category": "민사법"}, {"source_id": "ART003282943", "category": "공법"}, {"source_id": "ART003024313", "category": "기타"}, {"source_id": "ART003024316", "category


LLM 분류 진행 중:   0%|          | 0/96 [00:00<?, ?it/s]


--- 배치 1 시작 (논문 1~20) ---
📏 프롬프트 길이: 4518 글자
🔄 API 호출 시작... (배치 크기: 20)
✅ API 응답 완료 (18.62초)
✅ 저장 완료 (20건)


LLM 분류 진행 중:   1%|          | 1/96 [00:19<31:04, 19.62s/it]

...


--- 배치 10 시작 (논문 181~200) ---
📏 프롬프트 길이: 5518 글자
🔄 API 호출 시작... (배치 크기: 20)
✅ API 응답 완료 (22.67초)
✅ 저장 완료 (20건)


LLM 분류 진행 중:  10%|█         | 10/96 [03:42<31:47, 22.18s/it]


--- 배치 11 시작 (논문 201~220) ---
📏 프롬프트 길이: 5261 글자
🔄 API 호출 시작... (배치 크기: 20)
✅ API 응답 완료 (20.26초)
✅ 저장 완료 (20건)


LLM 분류 진행 중:  11%|█▏        | 11/96 [04:03<31:01, 21.90s/it]


--- 배치 12 시작 (논문 221~240) ---
📏 프롬프트 길이: 4987 글자
🔄 API 호출 시작... (배치 크기: 20)


### BERT Topic으로 분류

In [1]:
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd
import os

def perform_kobert_topic_modeling(file_path):
    # 1. 데이터 로드 및 전처리
    df = pd.read_csv(file_path)
    
    # 텍스트 결합 (NaN 처리 및 공백 제거)
    df['combined_text'] = (
        df['제목'].fillna('') + " " + 
        df['초록'].fillna('') + " " + 
        df['키워드'].fillna('')
    ).str.strip()
    
    # 유효한 텍스트가 있는 행만 추출
    df = df[df['combined_text'] != ""].reset_index(drop=True)
    docs = df['combined_text'].tolist()

    # 2. Embedding 모델 로드 (jhgan/ko-sbert-sts)
    model_name = 'jhgan/ko-sbert-sts' 
    print(f"📡 모델 로딩 중: {model_name}...")
    
    embedding_model = SentenceTransformer(model_name)
    embedding_model.max_seq_length = 512 

    # 3. 벡터라이저 설정 (불용어 필터링)
    vectorizer_model = CountVectorizer(
        ngram_range=(1, 2), 
        stop_words=['인공지능', '분석', '연구', '고찰', '대한', '있으며', '따라']
    )

    # 4. BERTopic 모델 생성 (4개 클러스터)
    topic_model = BERTopic(
        embedding_model=embedding_model,
        vectorizer_model=vectorizer_model,
        nr_topics=4, 
        verbose=True
    )

    # 5. 모델 학습 및 결과 할당
    print("🚀 BERTopic 학습 시작...")
    topics, probs = topic_model.fit_transform(docs)
    df['Cluster_ID'] = topics

    # 6. 결과 저장 (combined_text 칼럼 제거)
    topic_info = topic_model.get_topic_info()
    
    # --- 수정된 부분: 저장용 데이터프레임에서 불필요한 칼럼 제거 ---
    df_output = df.drop(columns=['combined_text'])
    
    df_output.to_csv('법학_AI_논문_BERTopic_클러스터링.csv', index=False, encoding='utf-8-sig')
    topic_info.to_csv('클러스터별_대표키워드_정보.csv', index=False, encoding='utf-8-sig')
    
    print("-" * 50)
    print("✅ 분석 완료! 'combined_text'를 제외한 결과가 저장되었습니다.")
    return df_output, topic_info, topic_model

if __name__ == "__main__":
    file_name = '법학_AI_논문_상세정보_리스트.csv'
    if os.path.exists(file_name):
        result_df, info, model = perform_kobert_topic_modeling(file_name)
    else:
        print(f"❌ 파일을 찾을 수 없습니다: {file_name}")
        
# 실행 전 커널을 한 번 Restart 하시는 것을 권장합니다!
result_df, info, model = perform_kobert_topic_modeling('법학_AI_논문_상세정보_리스트.csv')

📡 모델 로딩 중: jhgan/ko-sbert-sts...


2026-02-25 12:30:33,239 - BERTopic - Embedding - Transforming documents to embeddings.


🚀 BERTopic 학습 시작...


Batches:   0%|          | 0/60 [00:00<?, ?it/s]

2026-02-25 12:30:48,487 - BERTopic - Embedding - Completed ✓
2026-02-25 12:30:48,487 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-02-25 12:30:56,661 - BERTopic - Dimensionality - Completed ✓
2026-02-25 12:30:56,662 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-02-25 12:30:56,700 - BERTopic - Cluster - Completed ✓
2026-02-25 12:30:56,700 - BERTopic - Representation - Extracting topics using c-TF-IDF for topic reduction.
2026-02-25 12:30:58,352 - BERTopic - Representation - Completed ✓
2026-02-25 12:30:58,356 - BERTopic - Topic reduction - Reducing number of topics
2026-02-25 12:30:58,357 - BERTopic - Topic reduction - Number of topics (4) is equal or higher than the clustered topics(3).
2026-02-25 12:30:58,357 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-02-25 12:31:00,169 - BERTopic - Representation - Completed ✓


...

2026-02-25 12:31:03,401 - BERTopic - Embedding - Transforming documents to embeddings.


🚀 BERTopic 학습 시작...


Batches:   0%|          | 0/60 [00:00<?, ?it/s]

2026-02-25 12:31:17,264 - BERTopic - Embedding - Completed ✓
2026-02-25 12:31:17,264 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-02-25 12:31:19,874 - BERTopic - Dimensionality - Completed ✓
2026-02-25 12:31:19,874 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-02-25 12:31:19,908 - BERTopic - Cluster - Completed ✓
2026-02-25 12:31:19,909 - BERTopic - Representation - Extracting topics using c-TF-IDF for topic reduction.
2026-02-25 12:31:21,633 - BERTopic - Representation - Completed ✓
2026-02-25 12:31:21,637 - BERTopic - Topic reduction - Reducing number of topics
2026-02-25 12:31:21,646 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-02-25 12:31:23,444 - BERTopic - Representation - Completed ✓
2026-02-25 12:31:23,449 - BERTopic - Topic reduction - Reduced number of topics from 30 to 4


--------------------------------------------------
✅ 분석 완료! 'combined_text'를 제외한 결과가 저장되었습니다.


In [2]:
# 1. Cluster_ID별 빈도수 계산
cluster_counts = result_df['Cluster_ID'].value_counts().sort_index()

# 2. 비율(%) 계산
cluster_percentages = (cluster_counts / len(result_df)) * 100

# 3. 데이터프레임으로 깔끔하게 정리
summary_df = pd.DataFrame({
    '개수(건)': cluster_counts,
    '비율(%)': cluster_percentages.round(2)
})

# 인덱스 이름 설정 (Cluster_ID)
summary_df.index.name = 'Cluster ID'

print("=== 클러스터별 분포 요약 ===")
print(summary_df)

# (선택 사항) 요약 결과도 CSV로 저장하고 싶다면:
# summary_df.to_csv('클러스터_비율_통계.csv', encoding='utf-8-sig')

=== 클러스터별 분포 요약 ===
            개수(건)  비율(%)
Cluster ID              
-1            574  30.16
 0           1139  59.85
 1            135   7.09
 2             55   2.89


### 인용관계 기준 분석

In [6]:
import pandas as pd
import networkx as nx
import numpy as np
from sklearn.cluster import SpectralClustering
import os

def calculate_fixed_k_communities(filter_file, node_file, edge_file, k=4, output_file='KCI_AI_노드_커뮤니티_4개고정.csv'):
    print(f"0/3. 필터링 데이터 로드 중...")
    # 필터링 기준이 되는 논문 ID 리스트 로드
    df_filter = pd.read_csv(filter_file)
    valid_ids = set(df_filter['논문ID'].unique())
    print(f"   - 필터링 기준 논문 수: {len(valid_ids)}개")

    print(f"1/3. 데이터 로드 및 필터링 수행 중 (K={k})...")
    df_nodes = pd.read_csv(node_file)
    df_edges = pd.read_csv(edge_file)

    # 노드 필터링: node_id가 valid_ids에 속하는 경우만 유지
    df_nodes_filtered = df_nodes[df_nodes['node_id'].isin(valid_ids)].copy()
    filtered_node_ids = set(df_nodes_filtered['node_id'])
    
    # 엣지 필터링: source_id와 target_id_final 모두 valid_ids에 속하는 경우만 유지
    # (그래프의 무결성을 위해 양쪽 노드가 모두 존재해야 합니다)
    df_edges_filtered = df_edges[
        df_edges['source_id'].isin(filtered_node_ids) & 
        df_edges['target_id_final'].isin(filtered_node_ids)
    ].copy()

    print(f"   - 필터링 후 노드 수: {len(df_nodes_filtered)}개")
    print(f"   - 필터링 후 엣지 수: {len(df_edges_filtered)}개")

    # 그래프 생성 (무방향)
    G = nx.Graph()
    G.add_nodes_from(df_nodes_filtered['node_id'])
    edges = list(zip(df_edges_filtered['source_id'], df_edges_filtered['target_id_final']))
    G.add_edges_from(edges)

    # 노드 순서 보존을 위한 리스트
    node_list = list(G.nodes())
    
    # 2/3. 스펙트럴 클러스터링 실행
    if len(node_list) < k:
        print(f"⚠️ 경고: 필터링된 노드 수({len(node_list)})가 설정된 클러스터 수({k})보다 작습니다.")
        return None

    print(f"2/3. 스펙트럴 클러스터링 알고리즘 계산 중... (노드 {len(node_list)}개 대상)")
    adj_matrix = nx.to_numpy_array(G, nodelist=node_list)
    
    # n_clusters=k로 고정
    sc = SpectralClustering(
        n_clusters=k, 
        affinity='precomputed', 
        assign_labels='discretize', 
        random_state=42
    )
    labels = sc.fit_predict(adj_matrix)

    # 결과를 데이터프레임으로 변환
    df_partition = pd.DataFrame({'node_id': node_list, 'Community_ID': labels})

    print("3/3. 노드 메타데이터와 병합 및 저장 중...")
    # 필터링된 노드 정보와 병합 (원본 df_nodes가 아닌 df_nodes_filtered 사용)
    df_final = df_nodes_filtered.merge(df_partition, on='node_id', how='left')
    
    # 커뮤니티별 분포 확인
    print("\n📊 지정된 4개 커뮤니티별 노드 수:")
    print(df_final['Community_ID'].value_counts().sort_index())

    # 결과 저장
    df_final.to_csv(output_file, index=False, encoding='utf-8-sig')
    print(f"\n✅ 분석 완료! 결과 저장: {output_file}")

    return df_final

if __name__ == "__main__":
    # 파일 경로 설정
    filter_csv = '법학_AI_논문_상세정보_리스트.csv'
    node_csv = 'KCI_AI_전용_논문_노드.csv'
    edge_csv = 'KCI_AI_전용_인용_엣지.csv'
    
    if os.path.exists(filter_csv) and os.path.exists(node_csv) and os.path.exists(edge_csv):
        calculate_fixed_k_communities(filter_csv, node_csv, edge_csv, k=5)
    else:
        print("파일이 존재하지 않습니다. 경로를 확인해주세요.")

0/3. 필터링 데이터 로드 중...
   - 필터링 기준 논문 수: 1903개
1/3. 데이터 로드 및 필터링 수행 중 (K=5)...
   - 필터링 후 노드 수: 1903개
   - 필터링 후 엣지 수: 4977개
2/3. 스펙트럴 클러스터링 알고리즘 계산 중... (노드 1903개 대상)
3/3. 노드 메타데이터와 병합 및 저장 중...

📊 지정된 4개 커뮤니티별 노드 수:
Community_ID
0    1526
1     180
2       4
3     185
4       8
Name: count, dtype: int64

✅ 분석 완료! 결과 저장: KCI_AI_노드_커뮤니티_4개고정.csv


### 해상도 조절로 자동 분류

In [ ]:
# import pandas as pd
# import networkx as nx
# import community.community_louvain as community_louvain
# from itertools import combinations
# from collections import Counter
# import os
# import re

# def has_korean(text):
#     """한글이 한 글자라도 포함되어 있는지 확인하는 함수"""
#     return bool(re.search('[가-힣]', str(text)))

# def perform_optimized_clustering_korean_only(data_file, mapping_file, target_min=5, target_max=8):
#     if not os.path.exists(data_file) or not os.path.exists(mapping_file):
#         print("❌ 파일 경로를 확인해주세요.")
#         return

#     # 1. 매핑 사전 및 데이터 로드
#     df_map = pd.read_csv(mapping_file).dropna(subset=['Keyword', 'Target'])
#     mapping_dict = dict(zip(df_map[df_map['Type'] == '영문']['Keyword'].astype(str).str.lower(), 
#                             df_map[df_map['Type'] == '영문']['Target'].astype(str)))

#     df = pd.read_csv(data_file)
#     processed_docs = []
    
#     print("🧹 키워드 필터링 및 전처리 중 (한글 미포함 키워드 제외)...")
    
#     for k_str in df['키워드'].dropna():
#         # 기본 분리 및 소문자화
#         words = [w.strip().lower() for w in str(k_str).replace(';', ',').split(',') if w.strip()]
        
#         refined = []
#         for w in words:
#             # 1순위: 변환 표에 있으면 타겟어로 변경
#             final_w = mapping_dict.get(w, w)
            
#             # 2순위: 한글이 포함되어 있는지 확인 (필터링 핵심)
#             if has_korean(final_w) and str(final_w).lower() != 'nan':
#                 refined.append(str(final_w))
        
#         # 중복 제거
#         refined = list(set(refined))
        
#         # 키워드가 2개 이상인 경우만 네트워크 형성이 가능하므로 추가
#         if len(refined) >= 2:
#             processed_docs.append(refined)

#     # 2. 네트워크 생성
#     G = nx.Graph()
#     edge_counts = Counter()
#     for words in processed_docs:
#         for pair in combinations(sorted(words), 2):
#             edge_counts[pair] += 1
    
#     for (n1, n2), w in edge_counts.items():
#         if w >= 2: 
#             G.add_edge(n1, n2, weight=w)

#     if G.number_of_nodes() == 0:
#         print("⚠️ 한글 키워드가 포함된 관계가 없습니다.")
#         return

#     # 3. 해상도(Resolution) 자동 조절 루프 (5~8개 군집 목표)
#     print(f"🔍 적정 군집 개수({target_min}~{target_max}개) 찾는 중...")
#     res = 1.0
#     best_partition = None
    
#     for attempt in range(20):
#         partition = community_louvain.best_partition(G, weight='weight', resolution=res, random_state=42)
#         num_clusters = len(set(partition.values()))
        
#         if target_min <= num_clusters <= target_max:
#             best_partition = partition
#             break
#         elif num_clusters < target_min:
#             res *= 1.1 
#         else:
#             res *= 0.9 
    
#     best_partition = best_partition if best_partition else partition

#     # 4. 결과 정리 및 출력
#     clusters = {}
#     for node, c_id in best_partition.items():
#         clusters.setdefault(c_id, []).append(node)

#     print(f"\n✅ 분석 결과: {len(clusters)}개 군집 발견 (해상도: {res:.2f})")
    
#     summary_data = []
#     for c_id, nodes in sorted(clusters.items()):
#         # 연결 중심성이 높은 상위 키워드 추출
#         top_k = sorted(nodes, key=lambda x: G.degree(x), reverse=True)[:10]
#         nodes_str = ", ".join(top_k)
#         print(f"군집 {c_id+1}: {nodes_str}")
        
#         summary_data.append({
#             'Cluster_ID': c_id + 1,
#             'Keywords_Top10': nodes_str,
#             'Node_Count': len(nodes)
#         })

#     # 5. 저장
#     df_summary = pd.DataFrame(summary_data)
#     df_summary.to_csv('군집별_라벨링_작업표_한글전용.csv', index=False, encoding='utf-8-sig')
#     print("-" * 60)
#     print("📂 '군집별_라벨링_작업표_한글전용.csv' 저장 완료.")

#     return G, best_partition

# if __name__ == "__main__":
#     G, part = perform_optimized_clustering_korean_only('법학_AI_논문_상세정보_리스트.csv', '키워드 변환 표.csv')

🧹 키워드 필터링 및 전처리 중 (한글 미포함 키워드 제외)...
🔍 적정 군집 개수(5~8개) 찾는 중...

✅ 분석 결과: 115개 군집 발견 (해상도: 0.12)
군집 1: 인공지능, 생성형 인공지능, 개인정보, 저작권, 알고리즘, 4차 산업혁명, 빅데이터, 공정이용, 자율주행자동차, 제조물책임
군집 2: 편집저작물
군집 3: 국가기능
군집 4: 감사제도
군집 5: 전자소송, 전자문서, 신속절차, 디스커버리
군집 6: 디지털세
군집 7: 챗gpt, 창작적 기여
군집 8: 인공지능의 범죄능력, 인공지능의 형사책임능력
군집 9: 디지털 헬스케어, 보건권
군집 10: 양형, 재범예측
군집 11: 약인공지능
군집 12: 인간, 강인공지능, 뇌, 동물
군집 13: 공공영역
군집 14: 클라우드, 제4차 산업
군집 15: 인공지능 저작물의 저작자
군집 16: 데이터기반행정, ai 정부, 데이터 칸막이
군집 17: 보건증진, 산업발전
군집 18: 알고리즘 편향성, 예측적 치안
군집 19: 뇌과학, 신경과학
군집 20: 실시가능성
군집 21: 연구개발, 생명공학
군집 22: ai범죄, 배후정범
군집 23: 일자리, 기본소득제도
군집 24: 데이터 경제, 데이터 이동권
군집 25: 무인선박, 상업용 무인선박
군집 26: 묵시적 이용권, 옵트인
군집 27: 가상자산
군집 28: 변형적 이용, 데이터 마이닝, ai 학습, 시장 희석, 텍스트·데이터 마이닝, 법정허락, 스마트 계약, 텍스트데이터마이닝, 데이터마이닝, 비표현적 이용
군집 29: 원격조종 무인선박
군집 30: 저작권침해, 저작권 침해, 의거성
군집 31: 로보-어드바이저, 투자자문업
군집 32: 인권 보호
군집 33: 생체인식정보, 고위험 ai 시스템, ai 시스템
군집 34: 디지털의료제품법
군집 35: 설명가능한 인공지능
군집 36: eu 인공지능법, eu 제조물책임지침
군집 37: 소비자법, 집단소송
군집 38: 자동차손해배상보장법
군집 39: 변호사법, 변호사 광고
군집 40: 딥보이스
군집 41: 가명